In [6]:
from pyspark.sql.functions import*
from pyspark.sql.types import*

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 8, Finished, Available, Finished, False)

In [7]:
df_Company = spark.read.option("multiLine", True).json("Files/BRONZE/MASTER_DATA/company_master.json")
df_Currency = spark.read.option("multiLine", True).json("Files/BRONZE/MASTER_DATA/currency_exchange.json")
df_Plant = spark.read.option("multiLine", True).json("Files/BRONZE/MASTER_DATA/plant_master.json")
df_Purchase_Group = spark.read.option("multiLine", True).json("Files/BRONZE/MASTER_DATA/purchasing_group.json")
df_Cost_Center=spark.read.format("csv").option("header","True").load("Files/BRONZE/MASTER_DATA/cost_center_master.csv")
df_Date=spark.read.format("csv").option("header","True").load("Files/BRONZE/MASTER_DATA/date_dimension.csv")
df_Material=spark.read.format("csv").option("header","True").load("Files/BRONZE/MASTER_DATA/material_master.csv")
df_Vendor=spark.read.format("csv").option("header","True").load("Files/BRONZE/MASTER_DATA/vendor_master.csv")

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 9, Finished, Available, Finished, False)

In [8]:
display(df_Company)
display(df_Currency)
display(df_Plant)
display(df_Purchase_Group)
display(df_Cost_Center)
display(df_Date)
display(df_Material)
display(df_Vendor)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6bf731ad-628b-46be-88c8-f678e16002b7)

SynapseWidget(Synapse.DataFrame, a24dc2e6-1376-4dc4-8016-4e02e20a3e3e)

SynapseWidget(Synapse.DataFrame, 2682e7cb-99a0-4c23-8bf3-e5aa3fff00bf)

SynapseWidget(Synapse.DataFrame, 375df40e-e8fa-41ee-b37a-e8c7d74ce083)

SynapseWidget(Synapse.DataFrame, 35198abd-8068-4241-8cbe-5fdde024e46e)

SynapseWidget(Synapse.DataFrame, b4a28663-b1e5-4117-b667-a91419c5b682)

SynapseWidget(Synapse.DataFrame, 9a3663d0-63ee-4ab2-ade9-3566a07ac492)

SynapseWidget(Synapse.DataFrame, 5f02447b-050b-478c-b551-780234e3e915)

In [9]:
# User def Dropping Dulicates

def drop_duplicates(df, keys, table_name):
    before = df.count()
    df_clean = df.dropDuplicates(keys)
    after = df_clean.count()
    removed = before - after
    print("=" * 60)
    print(f"DUPLICATE ANALYSIS : {table_name}")
    print("=" * 60)
    print(f"{'Before':<15} | {before}")
    print(f"{'After':<15} | {after}")
    print(f"{'Removed':<15} | {removed}")
    print("=" * 60)
    return df_clean

df_Company       = drop_duplicates(df_Company,["Company_Code"],"Company")
df_Vendor        = drop_duplicates(df_Vendor,["Vendor_ID"],"Vendor")
df_Material      = drop_duplicates(df_Material,["Material_ID"],"Material")
df_Cost_Center   = drop_duplicates(df_Cost_Center,["Cost_Center_ID"],"Cost_Center")
df_Plant         = drop_duplicates(df_Plant,["Plant_ID"],"Plant")
df_Purchase_Group= drop_duplicates(df_Purchase_Group,["Purchasing_Group_ID"],"Purchase_Group")
df_Currency      = drop_duplicates(df_Currency,["Currency_Code"],"Currency")
df_Date          = drop_duplicates(df_Date,["Date_Key"],"Date")

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 11, Finished, Available, Finished, False)

DUPLICATE ANALYSIS : Company
Before          | 4
After           | 4
Removed         | 0
DUPLICATE ANALYSIS : Vendor
Before          | 236
After           | 236
Removed         | 0
DUPLICATE ANALYSIS : Material
Before          | 306
After           | 305
Removed         | 1
DUPLICATE ANALYSIS : Cost_Center
Before          | 147
After           | 146
Removed         | 1
DUPLICATE ANALYSIS : Plant
Before          | 10
After           | 10
Removed         | 0
DUPLICATE ANALYSIS : Purchase_Group
Before          | 20
After           | 20
Removed         | 0
DUPLICATE ANALYSIS : Currency
Before          | 4
After           | 4
Removed         | 0
DUPLICATE ANALYSIS : Date
Before          | 1461
After           | 1461
Removed         | 0


In [10]:
# Finding nulls_User defined

def find_nulls(df, table_name, key_columns):

    print("=" * 100)
    print(f"TABLE : {table_name}")
    print("=" * 100)

    for c in df.columns:

        null_count = df.filter(col(c).isNull()).count()

        if null_count > 0:

            print(f"\nColumn : {c} | Null Count : {null_count}")

            display(
                df.filter(col(c).isNull())
                  .select(*key_columns, c)
            )

    print()


    # Finding nulls

tables = {
    "Company": (df_Company, ["Company_Code"]),
    "Vendor": (df_Vendor, ["Vendor_ID"]),
    "Material": (df_Material, ["Material_ID"]),
    "Cost_Center": (df_Cost_Center, ["Cost_Center_ID"]),
    "Date": (df_Date, ["Date_Key"]),
    "Plant": (df_Plant, ["Plant_ID"]),
    "Purchase_Group": (df_Purchase_Group, ["Purchasing_Group_ID"]),
    "Currency": (df_Currency, ["Currency_Code"])
}

for table_name, (df, keys) in tables.items():
    find_nulls(df, table_name, keys)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 12, Finished, Available, Finished, False)

TABLE : Company

TABLE : Vendor

TABLE : Material

Column : Material_Group | Null Count : 1


SynapseWidget(Synapse.DataFrame, 86855c52-e200-4da5-a7db-1a015203cda5)


TABLE : Cost_Center

Column : Cost_Center_Name | Null Count : 1


SynapseWidget(Synapse.DataFrame, 60ec5635-d473-4463-a27d-72f612e57f17)


TABLE : Date

TABLE : Plant

TABLE : Purchase_Group

TABLE : Currency



In [11]:
print("=" * 60)
print("FILLING WITH UNKNOWN / DEFAULT VALUES")
print("=" * 60)


# ── Material ──────────────────────────────────────────────
df_Material = df_Material.fillna({"Material_Group" : "Services"})

# ── Cost Center ───────────────────────────────────────────
df_Cost_Center = df_Cost_Center.fillna({"Cost_Center_Name" : "Cost Center 0004"})


print("All default fills complete")

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 13, Finished, Available, Finished, False)

FILLING WITH UNKNOWN / DEFAULT VALUES
All default fills complete


In [12]:

def find_whitespace(df, table_name):
    print("=" * 60)
    print(f"WHITESPACE AUDIT : {table_name}")
    
    found = False
    for c in df.columns:
        # Compare length before and after trim
        diff = df.filter(
            length(col(c)) != length(trim(col(c)))
        ).count()
        if diff > 0:
            found = True
            print(f"{c:<30} | Rows with spaces: {diff}")
    if not found:
        print("No whitespace issues found ✓")
    print("=" * 60)

find_whitespace(df_Company,"Company")
find_whitespace(df_Vendor,"Vendor")
find_whitespace(df_Material,"Material")
find_whitespace(df_Cost_Center,"Cost_Center")
find_whitespace(df_Plant,"Plant")
find_whitespace(df_Purchase_Group,"Purchase_Group")
find_whitespace(df_Currency,"Currency")

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 14, Finished, Available, Finished, False)

WHITESPACE AUDIT : Company
No whitespace issues found ✓
WHITESPACE AUDIT : Vendor
Vendor_Name                    | Rows with spaces: 1
Country_Code                   | Rows with spaces: 1
WHITESPACE AUDIT : Material
Material_Name                  | Rows with spaces: 1
WHITESPACE AUDIT : Cost_Center
No whitespace issues found ✓
WHITESPACE AUDIT : Plant
No whitespace issues found ✓
WHITESPACE AUDIT : Purchase_Group
No whitespace issues found ✓
WHITESPACE AUDIT : Currency
No whitespace issues found ✓


In [25]:
## Trimming values

df_Vendor = df_Vendor.withColumn("Vendor_Name",trim(col("Vendor_Name")))
df_Vendor = df_Vendor.withColumn("Country_Code",trim(col("Country_Code")))
df_Material = df_Material.withColumn("Material_Name",trim(col("Material_Name")))

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 27, Finished, Available, Finished, False)

In [15]:
# Company
df_Company = df_Company.withColumn("Created_Date",to_date(col("Created_Date")))\
.withColumn("Last_Modified_Date", to_date(col("Last_Modified_Date")))

# Vendor
df_Vendor = df_Vendor.withColumn("Created_Date",to_date(col("Created_Date")))\
.withColumn("Last_Modified_Date", to_date(col("Last_Modified_Date")))

# Material
df_Material = df_Material.withColumn("Created_Date",to_date(col("Created_Date")))\
.withColumn("Last_Modified_Date", to_date(col("Last_Modified_Date")))

# Cost Center
df_Cost_Center = df_Cost_Center.withColumn("Created_Date",to_date(col("Created_Date")))\
.withColumn("Last_Modified_Date", to_date(col("Last_Modified_Date")))

# Plant
df_Plant = df_Plant.withColumn("Created_Date",to_date(col("Created_Date")))\
.withColumn("Last_Modified_Date", to_date(col("Last_Modified_Date")))

# Purchase Group
df_Purchase_Group = df_Purchase_Group.withColumn("Created_Date",to_date(col("Created_Date")))\
.withColumn("Last_Modified_Date", to_date(col("Last_Modified_Date")))

# Currency
df_Currency = df_Currency .withColumn("Created_Date",to_date(col("Created_Date")))\
.withColumn("Last_Modified_Date", to_date(col("Last_Modified_Date")))

# Date Dimension
df_Date = df_Date.withColumn("Date", to_date(col("Date")))

print("Date conversion complete for all tables")

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 17, Finished, Available, Finished, False)

Date conversion complete for all tables


In [16]:
#── Vendor — add Country_Name and Is_Active_Vendor ────────
df_Vendor = df_Vendor.withColumn("Vendor_Name",initcap(trim(col("Vendor_Name")))) \
.withColumn("Country_Name",
when(col("Country_Code") == "GB", "United Kingdom")
.when(col("Country_Code") == "DE", "Germany")
.when(col("Country_Code") == "FR", "France")
.when(col("Country_Code") == "US", "United States")
.when(col("Country_Code") == "IN", "India")
.when(col("Country_Code") == "NL", "Netherlands")
.otherwise("Other")) \
.withColumn("Is_Active_Vendor",
when(col("Vendor_Status") == "Active", "Yes")
.otherwise("No"))

print("Vendor transformed")


StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 18, Finished, Available, Finished, False)

Vendor transformed


In [17]:
df_Material = df_Material \
.withColumn("Material_Type_Desc",
when(col("Material_Type") == "DIEN", "Services")
.when(col("Material_Type") == "ROH",  "Raw Material")
.when(col("Material_Type") == "FERT", "Finished Goods")
.when(col("Material_Type") == "HALB", "Semi Finished")
.otherwise("Other")) \
.withColumn("Is_Active_Material",
when(col("Material_Status") == "Active", "Yes")
.otherwise("No"))

print("Material transformed")
display(df_Material)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 19, Finished, Available, Finished, False)

Material transformed


SynapseWidget(Synapse.DataFrame, 5e5fee03-68b3-4c01-a566-087c443a3968)

In [18]:
# ── Plant — add Country_Name ───────────────────────────────
df_Plant = df_Plant \
.withColumn("Country_Name",
when(col("Country") == "GB", "United Kingdom")
.when(col("Country") == "DE", "Germany")
.when(col("Country") == "FR", "France")
.when(col("Country") == "NL", "Netherlands")
.otherwise("Other"))

print("Plant transformed")
display(df_Plant)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 20, Finished, Available, Finished, False)

Plant transformed


SynapseWidget(Synapse.DataFrame, bceec183-b769-412a-827a-b7e2e564fd93)

In [19]:
df_Currency = df_Currency \
.withColumn("Exchange_Rate_To_GBP",
col("Exchange_Rate_To_GBP").cast("double")) \
.withColumn("Currency_Strength",
when(col("Exchange_Rate_To_GBP") >= 1.0, "Strong")
.when(col("Exchange_Rate_To_GBP") >= 0.5, "Medium")
.otherwise("Weak")) \
.withColumn("Is_Base_Currency",
when(col("Currency_Code") == "GBP", "Yes")
.otherwise("No"))

print("Currency transformed")
display(df_Currency)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 21, Finished, Available, Finished, False)

Currency transformed


SynapseWidget(Synapse.DataFrame, fd10bec5-d9db-46b8-b3f4-ba1e979b349f)

In [20]:
# ── Purchase Group — add Dept_Category ────────────────────
df_Purchase_Group = df_Purchase_Group \
.withColumn("Dept_Category",
when(col("Department") == "IT Procurement","IT")
.when(col("Department") == "MRO Procurement","MRO")
.when(col("Department") == "Direct Materials","Direct")
.when(col("Department") == "Services","Indirect")
.when(col("Department") == "Facilities","Facilities")
.otherwise("Other"))

print("Purchase Group transformed")
display(df_Purchase_Group)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 22, Finished, Available, Finished, False)

Purchase Group transformed


SynapseWidget(Synapse.DataFrame, f0bb14bc-ccb1-45ed-82bc-e6e62f2d307d)

In [21]:
# ── Date — add Is_Weekend, Financial_Quarter ───────────────
from pyspark.sql.functions import last_day

df_Date = df_Date\
.withColumn("Is_Weekend",
when(col("Weekday_Name").isin("Saturday", "Sunday"), "Yes").otherwise("No"))\
.withColumn("Is_Month_End",when(col("Date") == last_day(col("Date")), "Yes").otherwise("No"))\
.withColumn("Financial_Quarter",when(col("Month").isin(4, 5, 6),"Q1")
.when(col("Month").isin(7, 8, 9),"Q2")
.when(col("Month").isin(10, 11, 12),"Q3")
.otherwise("Q4"))\
.withColumn("Season",
when(col("Month").isin(12, 1, 2),"Winter")
.when(col("Month").isin(3, 4, 5),"Spring")
.when(col("Month").isin(6, 7, 8),"Summer")
.otherwise("Autumn"))

print("Date transformed")
display(df_Date)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 23, Finished, Available, Finished, False)

Date transformed


SynapseWidget(Synapse.DataFrame, 24ab3fd6-a097-4a27-b659-af20d19ed060)

In [22]:
df_Company=df_Company.withColumn("LoadDate", current_timestamp())
df_Currency=df_Currency.withColumn("LoadDate", current_timestamp())
df_Plant=df_Plant.withColumn("LoadDate", current_timestamp())
df_Purchase_Group=df_Purchase_Group.withColumn("LoadDate", current_timestamp())
df_Cost_Center=df_Cost_Center.withColumn("LoadDate", current_timestamp())
df_Date=df_Date.withColumn("LoadDate", current_timestamp())
df_Material=df_Material.withColumn("LoadDate", current_timestamp())
df_Vendor=df_Vendor.withColumn("LoadDate", current_timestamp())
display(df_Plant)

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, db4f56e7-5581-4a24-b769-e081a0e32e1e)

In [ ]:
(df_Company).printSchema()
(df_Currency).printSchema()
(df_Plant).printSchema()
(df_Purchase_Group).printSchema()
(df_Cost_Center).printSchema()
(df_Date).printSchema()
(df_Material).printSchema()
(df_Vendor).printSchema()

In [23]:
# Writing cleaned and transformed data into Delta format
# Each cleaned and transformed table saved separately
df_Vendor.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_vendor")
df_Material.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_material")
df_Cost_Center.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_cost_center")
df_Company.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_company")
df_Plant.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_plant")
df_Purchase_Group.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_purchase_group")
df_Currency.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_currency")
df_Date.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable("silver_dim_date")

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 25, Finished, Available, Finished, False)

In [24]:
df_Vendor.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_vendor")
df_Material.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_material")
df_Cost_Center.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_cost_center")
df_Company.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_company")
df_Plant.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_plant")
df_Purchase_Group.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_purchase_group")
df_Currency.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_currency")
df_Date.write.mode("overwrite").parquet("Files/SILVER/DIMENSION/silver_date")

StatementMeta(, e504e4e9-09dd-4aa4-83ec-8deb2508c213, 26, Finished, Available, Finished, False)